# 08 — Exploração, Tratamento e Integração dos Dados SEEG

## Projeto AgroESG — Soja | Centro-Oeste e Sul

Este notebook audita, trata e prepara os dados do SEEG relacionados
às emissões associadas à cultura da soja para posterior integração
com a base agroambiental construída nos notebooks anteriores.

O recorte analisado corresponde às emissões de **N₂O associadas aos
resíduos agrícolas da soja em solos manejados**, considerando emissões
diretas e indiretas por lixiviação/escorrimento superficial.

Para representação em CO₂ equivalente, será utilizado como referência
analítica o indicador **CO2e (t) GWP-AR6**.

## Escopo

- Cultura: soja
- Regiões: Centro-Oeste e Sul
- UFs: DF, GO, MT, MS, PR, RS e SC
- Período: 2019–2024
- Fonte: SEEG
- Gás de origem: N₂O
- Indicador principal: CO₂e GWP-AR6

## Objetivos

1. Auditar os arquivos Curated produzidos a partir do SEEG;
2. Validar estrutura, granularidade, período e recorte territorial;
3. Tratar `dados_nao_captados` no nível correto de ano;
4. Transformar os dados de formato wide para long;
5. Consolidar emissões diretas e indiretas;
6. Consolidar fragmentos de bioma no nível municipal;
7. Associar o código IBGE;
8. Produzir uma camada SEEG no nível `municipio × ano`;
9. Preparar a base para integração posterior com a base agroambiental.

> Os indicadores utilizados neste notebook representam um recorte
> específico das emissões associadas aos resíduos agrícolas da soja.
> Eles não representam todas as emissões do ciclo de vida da soja.

In [1]:
# ============================================================
# IMPORTAÇÕES E DIRETÓRIOS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Diretório raiz do projeto
# ------------------------------------------------------------

BASE_DIR = Path(
    r"C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao"
)


# ------------------------------------------------------------
# Diretórios SEEG produzidos pela integrante da equipe
# ------------------------------------------------------------

SEEG_CURATED_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "seeg"
    / "databases_curated"
)


SEEG_PROCESSED_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "seeg"
    / "databases_processed"
)


print("BASE_DIR:")
print(BASE_DIR)

print("\nSEEG Curated:")
print(SEEG_CURATED_DIR)

print("\nSEEG Processed:")
print(SEEG_PROCESSED_DIR)

print("\nDiretórios existem?")
print("Curated:", SEEG_CURATED_DIR.exists())
print("Processed:", SEEG_PROCESSED_DIR.exists())

BASE_DIR:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao

SEEG Curated:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\seeg\databases_curated

SEEG Processed:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\seeg\databases_processed

Diretórios existem?
Curated: True
Processed: True


In [2]:
# ============================================================
# INVENTÁRIO DOS ARQUIVOS SEEG
# ============================================================

arquivos_curated_seeg = sorted(
    SEEG_CURATED_DIR.glob("*.csv")
)

arquivos_processed_seeg = sorted(
    SEEG_PROCESSED_DIR.glob("*.csv")
)


print("=" * 70)
print("ARQUIVOS CURATED")
print("=" * 70)

for arquivo in arquivos_curated_seeg:

    tamanho_mb = (
        arquivo.stat().st_size
        / (1024 * 1024)
    )

    print(
        f"{arquivo.name} -> {tamanho_mb:.2f} MB"
    )


print("\n" + "=" * 70)
print("ARQUIVOS PROCESSED")
print("=" * 70)

for arquivo in arquivos_processed_seeg:

    tamanho_mb = (
        arquivo.stat().st_size
        / (1024 * 1024)
    )

    print(
        f"{arquivo.name} -> {tamanho_mb:.2f} MB"
    )

ARQUIVOS CURATED
soja_ar2_2019_2024.csv -> 1.47 MB
soja_ar4_2019_2024.csv -> 1.47 MB
soja_ar5_2019_2024.csv -> 1.47 MB
soja_ar6_2019_2024.csv -> 1.47 MB
soja_emissoes_2019_2024_com_dados_nao_captados.csv -> 6.67 MB
soja_emissoes_2019_2024_sem_dados_nao_captados.csv -> 6.66 MB
soja_gases_2019_2024.csv -> 0.66 MB

ARQUIVOS PROCESSED
ar2_processed.csv -> 435.73 MB
ar4_processed.csv -> 435.69 MB
ar5_processed.csv -> 435.17 MB
ar6_processed.csv -> 435.46 MB
gases_processed.csv -> 384.92 MB


In [3]:
# ============================================================
# ARQUIVOS CURATED PRINCIPAIS
# ============================================================

ARQUIVO_SEEG_AR6 = (
    SEEG_CURATED_DIR
    / "soja_ar6_2019_2024.csv"
)


ARQUIVO_SEEG_GASES = (
    SEEG_CURATED_DIR
    / "soja_gases_2019_2024.csv"
)


print("AR6:")
print(ARQUIVO_SEEG_AR6)

print("\nGases:")
print(ARQUIVO_SEEG_GASES)

print("\nArquivos encontrados:")
print("AR6:", ARQUIVO_SEEG_AR6.exists())
print("Gases:", ARQUIVO_SEEG_GASES.exists())

AR6:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\seeg\databases_curated\soja_ar6_2019_2024.csv

Gases:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\seeg\databases_curated\soja_gases_2019_2024.csv

Arquivos encontrados:
AR6: True
Gases: True


In [4]:
# ============================================================
# CARREGAMENTO DOS CURATED SEEG
# ============================================================

seeg_ar6 = pd.read_csv(
    ARQUIVO_SEEG_AR6,
    low_memory=False
)


seeg_gases = pd.read_csv(
    ARQUIVO_SEEG_GASES,
    low_memory=False
)


print(
    "SEEG AR6:"
)

print(
    seeg_ar6.shape
)


print(
    "\nSEEG Gases:"
)

print(
    seeg_gases.shape
)

SEEG AR6:
(7684, 18)

SEEG Gases:
(3842, 18)


In [5]:
# ============================================================
# INSPEÇÃO INICIAL DAS COLUNAS
# ============================================================

print(
    "COLUNAS — AR6"
)

print(
    seeg_ar6.columns.tolist()
)


print(
    "\nCOLUNAS — GASES"
)

print(
    seeg_gases.columns.tolist()
)


print(
    "\nPrimeiros registros AR6:"
)

display(
    seeg_ar6.head()
)


print(
    "\nPrimeiros registros Gases:"
)

display(
    seeg_gases.head()
)

COLUNAS — AR6
['setor_de_emissao', 'categoria_emissora', 'sub_categoria_emissora', 'produto_ou_sistema', 'detalhamento', 'recorte', 'atividade_geral', 'bioma', 'emissao_remocao_bunker', 'gas', '2019', '2020', '2021', '2022', '2023', '2024', 'municipio', 'estado']

COLUNAS — GASES
['setor_de_emissao', 'categoria_emissora', 'sub_categoria_emissora', 'produto_ou_sistema', 'detalhamento', 'recorte', 'atividade_geral', 'bioma', 'emissao_remocao_bunker', 'gas', '2019', '2020', '2021', '2022', '2023', '2024', 'municipio', 'estado']

Primeiros registros AR6:


,setor_de_emissao,categoria_emissora,sub_categoria_emissora,produto_ou_sistema,detalhamento,recorte,atividade_geral,bioma,emissao_remocao_bunker,gas,2019,2020,2021,2022,2023,2024,municipio,estado
0,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GTP-AR6,17268.36,19223.45,20642.63,20055.11,20245.66,19233.38,Brasília,DF
1,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,20232.88,22523.62,24186.43,23498.05,23721.31,22535.24,Brasília,DF
2,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Indiretas (lixiviação/escorrimento superficial),Agricultura,Cerrado,Emissão,CO2e (t) GTP-AR6,3885.38,4325.28,4644.59,4512.4,4555.27,4327.51,Brasília,DF
3,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Indiretas (lixiviação/escorrimento superficial),Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,4552.4,5067.81,5441.95,5287.06,5337.29,5070.43,Brasília,DF
4,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GTP-AR6,12.9,59.55,141.92,190.55,347.62,191.21,Abadia de Goiás,GO



Primeiros registros Gases:


,setor_de_emissao,categoria_emissora,sub_categoria_emissora,produto_ou_sistema,detalhamento,recorte,atividade_geral,bioma,emissao_remocao_bunker,gas,2019,2020,2021,2022,2023,2024,municipio,estado
0,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,N2O (t),74.11,82.5,88.59,86.07,86.89,82.55,Brasília,DF
1,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Indiretas (lixiviação/escorrimento superficial),Agricultura,Cerrado,Emissão,N2O (t),16.68,18.56,19.93,19.37,19.55,18.57,Brasília,DF
2,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,N2O (t),0.06,0.26,0.61,0.82,1.49,0.82,Abadia de Goiás,GO
3,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,N2O (t),6.73,5.86,3.87,4.97,7.16,7.95,Abadiânia,GO
4,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,N2O (t),36.05,50.7,53.58,53.58,57.92,57.73,Acreúna,GO


In [6]:
# ============================================================
# AUDITORIA DE DOMÍNIO — SEEG AR6 E GASES
# ============================================================

COLUNAS_CATEGORICAS = [
    "setor_de_emissao",
    "categoria_emissora",
    "sub_categoria_emissora",
    "produto_ou_sistema",
    "detalhamento",
    "recorte",
    "atividade_geral",
    "bioma",
    "emissao_remocao_bunker",
    "gas",
    "estado"
]


for nome_base, base in {
    "SEEG AR6": seeg_ar6,
    "SEEG GASES": seeg_gases
}.items():

    print("\n" + "=" * 70)
    print(nome_base)
    print("=" * 70)

    print("Dimensão:")
    print(base.shape)

    for coluna in COLUNAS_CATEGORICAS:

        print(f"\n{coluna}:")

        display(
            base[coluna]
            .value_counts(
                dropna=False
            )
        )


SEEG AR6
Dimensão:
(7684, 18)

setor_de_emissao:


setor_de_emissao
Agropecuária    7684
Name: count, dtype: int64


categoria_emissora:


categoria_emissora
Solos manejados    7684
Name: count, dtype: int64


sub_categoria_emissora:


...

emissao_remocao_bunker
Emissão    3842
Name: count, dtype: int64


gas:


gas
N2O (t)    3842
Name: count, dtype: int64


estado:


estado
RS    1274
PR     818
SC     590
GO     528
MT     408
MS     222
DF       2
Name: count, dtype: int64

In [7]:
# ============================================================
# AUDITORIA — DADOS NÃO CAPTADOS
# ============================================================

ANOS_FOCO = [
    "2019",
    "2020",
    "2021",
    "2022",
    "2023",
    "2024"
]


def auditar_dados_nao_captados(
    nome,
    base
):

    mascara = (
        base[
            ANOS_FOCO
        ]
        .astype("string")
        .eq(
            "dados_nao_captados"
        )
    )

    linhas_afetadas = (
        mascara
        .any(axis=1)
    )


    print("\n" + "=" * 70)
    print(nome)
    print("=" * 70)

    print(
        "Linhas com pelo menos um ano não captado:",
        linhas_afetadas.sum()
    )

    print(
        "Quantidade total de células não captadas:",
        mascara.sum().sum()
    )

    print(
        "Municípios afetados:"
    )

    display(
        base.loc[
            linhas_afetadas,
            [
                "municipio",
                "estado"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "estado",
                "municipio"
            ]
        )
    )


auditar_dados_nao_captados(
    "SEEG AR6",
    seeg_ar6
)

auditar_dados_nao_captados(
    "SEEG GASES",
    seeg_gases
)


SEEG AR6
Linhas com pelo menos um ano não captado: 12
Quantidade total de células não captadas: 60
Municípios afetados:


,municipio,estado
4814,Cerro Branco,RS
4919,Portão,RS
4993,Venâncio Aires,RS



SEEG GASES
Linhas com pelo menos um ano não captado: 6
Quantidade total de células não captadas: 30
Municípios afetados:


,municipio,estado
2429,Cerro Branco,RS
2534,Portão,RS
2608,Venâncio Aires,RS


In [8]:
# ============================================================
# SELEÇÃO DA MÉTRICA PRINCIPAL — GWP AR6
# ============================================================

seeg_ar6_gwp = (
    seeg_ar6[
        seeg_ar6[
            "gas"
        ].eq(
            "CO2e (t) GWP-AR6"
        )
    ]
    .copy()
)


print(
    "AR6 completo:"
)

print(
    seeg_ar6.shape
)


print(
    "\nAR6 somente GWP:"
)

print(
    seeg_ar6_gwp.shape
)


print(
    "\nValores da coluna gas:"
)

display(
    seeg_ar6_gwp[
        "gas"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nRecortes:"
)

display(
    seeg_ar6_gwp[
        "recorte"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nMunicípios:"
)

print(
    seeg_ar6_gwp[
        [
            "municipio",
            "estado"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

AR6 completo:
(7684, 18)

AR6 somente GWP:
(3842, 18)

Valores da coluna gas:


gas
CO2e (t) GWP-AR6    3842
Name: count, dtype: int64


Recortes:


recorte
Diretas                                            1921
Indiretas (lixiviação/escorrimento superficial)    1921
Name: count, dtype: int64


Municípios:
1658


In [9]:
# ============================================================
# TRANSFORMAÇÃO WIDE → LONG — CO2e GWP-AR6
# ============================================================

COLUNAS_IDENTIFICACAO_SEEG = [
    "setor_de_emissao",
    "categoria_emissora",
    "sub_categoria_emissora",
    "produto_ou_sistema",
    "detalhamento",
    "recorte",
    "atividade_geral",
    "bioma",
    "emissao_remocao_bunker",
    "gas",
    "municipio",
    "estado"
]


seeg_ar6_long = (
    seeg_ar6_gwp
    .melt(
        id_vars=COLUNAS_IDENTIFICACAO_SEEG,
        value_vars=ANOS_FOCO,
        var_name="ano",
        value_name="valor_original_co2e"
    )
)


seeg_ar6_long[
    "ano"
] = (
    seeg_ar6_long[
        "ano"
    ]
    .astype(int)
)


print(
    "Dimensão antes:"
)

print(
    seeg_ar6_gwp.shape
)


print(
    "\nDimensão long:"
)

print(
    seeg_ar6_long.shape
)


print(
    "\nAnos:"
)

print(
    sorted(
        seeg_ar6_long[
            "ano"
        ]
        .unique()
        .tolist()
    )
)


display(
    seeg_ar6_long.head(10)
)

Dimensão antes:
(3842, 18)

Dimensão long:
(23052, 14)

Anos:
[2019, 2020, 2021, 2022, 2023, 2024]


,setor_de_emissao,categoria_emissora,sub_categoria_emissora,produto_ou_sistema,detalhamento,recorte,atividade_geral,bioma,emissao_remocao_bunker,gas,municipio,estado,ano,valor_original_co2e
0,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,Brasília,DF,2019,20232.88
1,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Indiretas (lixiviação/escorrimento superficial),Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,Brasília,DF,2019,4552.4
2,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,Abadia de Goiás,GO,2019,15.12
3,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,Abadiânia,GO,2019,1838.56
4,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,Acreúna,GO,2019,9842.02
5,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,Adelândia,GO,2019,34.88
6,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,Água Fria de Goiás,GO,2019,8930.38
7,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,Água Limpa,GO,2019,0.0
8,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,Águas Lindas de Goiás,GO,2019,106.05
9,Agropecuária,Solos manejados,Resíduos agrícolas,Soja,Vegetal,Diretas,Agricultura,Cerrado,Emissão,CO2e (t) GWP-AR6,Alexânia,GO,2019,1368.86


In [10]:
# ============================================================
# TRATAMENTO — DADOS NÃO CAPTADOS
# ============================================================

seeg_ar6_long[
    "dado_nao_captado"
] = (
    seeg_ar6_long[
        "valor_original_co2e"
    ]
    .astype("string")
    .eq(
        "dados_nao_captados"
    )
)


seeg_ar6_long[
    "emissao_co2e_gwp_ar6_t"
] = pd.to_numeric(
    seeg_ar6_long[
        "valor_original_co2e"
    ],
    errors="coerce"
)


print(
    "Linhas totais:"
)

print(
    len(
        seeg_ar6_long
    )
)


print(
    "\nCélulas originalmente não captadas:"
)

print(
    seeg_ar6_long[
        "dado_nao_captado"
    ].sum()
)


print(
    "\nValores numéricos disponíveis:"
)

print(
    seeg_ar6_long[
        "emissao_co2e_gwp_ar6_t"
    ]
    .notna()
    .sum()
)


print(
    "\nValores ausentes após conversão:"
)

print(
    seeg_ar6_long[
        "emissao_co2e_gwp_ar6_t"
    ]
    .isna()
    .sum()
)

Linhas totais:
23052

Células originalmente não captadas:
30

Valores numéricos disponíveis:
23022

Valores ausentes após conversão:
30


In [11]:
# ============================================================
# AUDITORIA DO CASO CRÍTICO — CERRO BRANCO / RS
# ============================================================

auditoria_cerro_branco = (
    seeg_ar6_long[
        (
            seeg_ar6_long[
                "municipio"
            ].eq(
                "Cerro Branco"
            )
        )
        &
        (
            seeg_ar6_long[
                "estado"
            ].eq(
                "RS"
            )
        )
    ]
    [
        [
            "municipio",
            "estado",
            "bioma",
            "recorte",
            "ano",
            "valor_original_co2e",
            "dado_nao_captado",
            "emissao_co2e_gwp_ar6_t"
        ]
    ]
    .sort_values(
        [
            "bioma",
            "recorte",
            "ano"
        ]
    )
)


display(
    auditoria_cerro_branco
)

,municipio,estado,bioma,recorte,ano,valor_original_co2e,dado_nao_captado,emissao_co2e_gwp_ar6_t
2048,Cerro Branco,RS,Mata Atlântica,Diretas,2019,282.95,False,282.95
5890,Cerro Branco,RS,Mata Atlântica,Diretas,2020,167.44,False,167.44
9732,Cerro Branco,RS,Mata Atlântica,Diretas,2021,332.56,False,332.56
13574,Cerro Branco,RS,Mata Atlântica,Diretas,2022,273.49,False,273.49
17416,Cerro Branco,RS,Mata Atlântica,Diretas,2023,182.32,False,182.32
21258,Cerro Branco,RS,Mata Atlântica,Diretas,2024,312.63,False,312.63
2685,Cerro Branco,RS,Mata Atlântica,Indiretas (lixiviação/escorrimento superficial),2019,63.66,False,63.66
6527,Cerro Branco,RS,Mata Atlântica,Indiretas (lixiviação/escorrimento superficial),2020,37.68,False,37.68
10369,Cerro Branco,RS,Mata Atlântica,Indiretas (lixiviação/escorrimento superficial),2021,74.83,False,74.83
14211,Cerro Branco,RS,Mata Atlântica,Indiretas (lixiviação/escorrimento superficial),2022,61.53,False,61.53


In [13]:
# ============================================================
# TRANSFORMAÇÃO WIDE → LONG — N2O
# ============================================================

seeg_gases_long = (
    seeg_gases
    .melt(
        id_vars=COLUNAS_IDENTIFICACAO_SEEG,
        value_vars=ANOS_FOCO,
        var_name="ano",
        value_name="valor_original_n2o"
    )
)


seeg_gases_long[
    "ano"
] = (
    seeg_gases_long[
        "ano"
    ]
    .astype(int)
)


print(
    "Dimensão antes:"
)

print(
    seeg_gases.shape
)


print(
    "\nDimensão long:"
)

print(
    seeg_gases_long.shape
)


print(
    "\nAnos:"
)

print(
    sorted(
        seeg_gases_long[
            "ano"
        ]
        .unique()
        .tolist()
    )
)

Dimensão antes:
(3842, 18)

Dimensão long:
(23052, 14)

Anos:
[2019, 2020, 2021, 2022, 2023, 2024]


In [14]:
# ============================================================
# TRATAMENTO — N2O NÃO CAPTADO
# ============================================================

seeg_gases_long[
    "dado_nao_captado_n2o"
] = (
    seeg_gases_long[
        "valor_original_n2o"
    ]
    .astype("string")
    .eq(
        "dados_nao_captados"
    )
)


seeg_gases_long[
    "emissao_n2o_t"
] = pd.to_numeric(
    seeg_gases_long[
        "valor_original_n2o"
    ],
    errors="coerce"
)


print(
    "Linhas totais:"
)

print(
    len(
        seeg_gases_long
    )
)


print(
    "\nCélulas originalmente não captadas:"
)

print(
    seeg_gases_long[
        "dado_nao_captado_n2o"
    ].sum()
)


print(
    "\nValores numéricos:"
)

print(
    seeg_gases_long[
        "emissao_n2o_t"
    ]
    .notna()
    .sum()
)


print(
    "\nValores ausentes:"
)

print(
    seeg_gases_long[
        "emissao_n2o_t"
    ]
    .isna()
    .sum()
)

Linhas totais:
23052

Células originalmente não captadas:
30

Valores numéricos:
23022

Valores ausentes:
30


In [15]:
# ============================================================
# CHAVE DE CORRESPONDÊNCIA — N2O × CO2e
# ============================================================

CHAVE_SEEG_DETALHADA = [
    "setor_de_emissao",
    "categoria_emissora",
    "sub_categoria_emissora",
    "produto_ou_sistema",
    "detalhamento",
    "recorte",
    "atividade_geral",
    "bioma",
    "emissao_remocao_bunker",
    "municipio",
    "estado",
    "ano"
]


print(
    "Duplicatas AR6:"
)

print(
    seeg_ar6_long
    .duplicated(
        subset=CHAVE_SEEG_DETALHADA
    )
    .sum()
)


print(
    "\nDuplicatas Gases:"
)

print(
    seeg_gases_long
    .duplicated(
        subset=CHAVE_SEEG_DETALHADA
    )
    .sum()
)

Duplicatas AR6:
0

Duplicatas Gases:
0


In [16]:
# ============================================================
# INTEGRAÇÃO DETALHADA — N2O × CO2e GWP-AR6
# ============================================================

seeg_detalhado = (
    seeg_ar6_long[
        CHAVE_SEEG_DETALHADA
        +
        [
            "dado_nao_captado",
            "emissao_co2e_gwp_ar6_t"
        ]
    ]
    .merge(
        seeg_gases_long[
            CHAVE_SEEG_DETALHADA
            +
            [
                "dado_nao_captado_n2o",
                "emissao_n2o_t"
            ]
        ],
        on=CHAVE_SEEG_DETALHADA,
        how="left",
        validate="one_to_one"
    )
)


print(
    "Dimensão:"
)

print(
    seeg_detalhado.shape
)


print(
    "\nDuplicatas na chave detalhada:"
)

print(
    seeg_detalhado
    .duplicated(
        subset=CHAVE_SEEG_DETALHADA
    )
    .sum()
)


print(
    "\nNulos CO2e:"
)

print(
    seeg_detalhado[
        "emissao_co2e_gwp_ar6_t"
    ]
    .isna()
    .sum()
)


print(
    "\nNulos N2O:"
)

print(
    seeg_detalhado[
        "emissao_n2o_t"
    ]
    .isna()
    .sum()
)

Dimensão:
(23052, 16)

Duplicatas na chave detalhada:
0

Nulos CO2e:
30

Nulos N2O:
30


In [17]:
# ============================================================
# CLASSIFICAÇÃO DOS RECORTES DE EMISSÃO
# ============================================================

MAPA_TIPO_EMISSAO = {
    "Diretas": "direta",
    "Indiretas (lixiviação/escorrimento superficial)": "indireta"
}


seeg_detalhado[
    "tipo_emissao"
] = (
    seeg_detalhado[
        "recorte"
    ]
    .map(
        MAPA_TIPO_EMISSAO
    )
)


print(
    "Tipos de emissão:"
)

display(
    seeg_detalhado[
        "tipo_emissao"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nTipos não reconhecidos:"
)

print(
    seeg_detalhado[
        "tipo_emissao"
    ]
    .isna()
    .sum()
)

Tipos de emissão:


tipo_emissao
direta      11526
indireta    11526
Name: count, dtype: int64


Tipos não reconhecidos:
0


In [21]:
# ============================================================
# CONSOLIDAÇÃO CORRIGIDA — MUNICÍPIO × BIOMA × ANO
# ============================================================

CHAVE_SEEG_BIOMA_ANO = [
    "municipio",
    "estado",
    "bioma",
    "ano"
]


# ------------------------------------------------------------
# Auditoria antes do pivot
# Deve existir somente uma linha por:
# município + estado + bioma + ano + tipo_emissao
# ------------------------------------------------------------

duplicatas_pre_pivot = (
    seeg_detalhado
    .duplicated(
        subset=
        CHAVE_SEEG_BIOMA_ANO
        +
        [
            "tipo_emissao"
        ]
    )
    .sum()
)


print(
    "Duplicatas antes do pivot:"
)

print(
    duplicatas_pre_pivot
)


# ------------------------------------------------------------
# Pivot somente das combinações realmente observadas
# ------------------------------------------------------------

seeg_bioma_ano = (
    seeg_detalhado
    .pivot(
        index=CHAVE_SEEG_BIOMA_ANO,
        columns="tipo_emissao",
        values=[
            "emissao_co2e_gwp_ar6_t",
            "emissao_n2o_t"
        ]
    )
    .reset_index()
)


# ------------------------------------------------------------
# Achatar MultiIndex das colunas
# ------------------------------------------------------------

seeg_bioma_ano.columns = [
    "_".join(
        [
            str(parte)
            for parte in coluna
            if str(parte) != ""
        ]
    )
    if isinstance(
        coluna,
        tuple
    )
    else coluna
    for coluna in seeg_bioma_ano.columns
]


print(
    "\nDimensão:"
)

print(
    seeg_bioma_ano.shape
)


print(
    "\nColunas:"
)

print(
    seeg_bioma_ano.columns.tolist()
)


print(
    "\nCombinações município + estado + bioma:"
)

print(
    seeg_bioma_ano[
        [
            "municipio",
            "estado",
            "bioma"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


display(
    seeg_bioma_ano.head(10)
)

Duplicatas antes do pivot:
0

Dimensão:
(11526, 8)

Colunas:
['municipio', 'estado', 'bioma', 'ano', 'emissao_co2e_gwp_ar6_t_direta', 'emissao_co2e_gwp_ar6_t_indireta', 'emissao_n2o_t_direta', 'emissao_n2o_t_indireta']

Combinações município + estado + bioma:
1921


,municipio,estado,bioma,ano,emissao_co2e_gwp_ar6_t_direta,emissao_co2e_gwp_ar6_t_indireta,emissao_n2o_t_direta,emissao_n2o_t_indireta
0,Abadia de Goiás,GO,Cerrado,2019,15.12,3.40,0.06,0.01
1,Abadia de Goiás,GO,Cerrado,2020,69.77,15.70,0.26,0.06
2,Abadia de Goiás,GO,Cerrado,2021,166.28,37.41,0.61,0.14
3,Abadia de Goiás,GO,Cerrado,2022,223.26,50.23,0.82,0.18
4,Abadia de Goiás,GO,Cerrado,2023,407.29,91.64,1.49,0.34
5,Abadia de Goiás,GO,Cerrado,2024,224.03,50.41,0.82,0.18
6,Abadiânia,GO,Cerrado,2019,1838.56,413.68,6.73,1.52
7,Abadiânia,GO,Cerrado,2020,1601.11,360.25,5.86,1.32
8,Abadiânia,GO,Cerrado,2021,1056.61,237.74,3.87,0.87
9,Abadiânia,GO,Cerrado,2022,1356.61,305.24,4.97,1.12


In [24]:
# ============================================================
# STATUS E TOTAL — MUNICÍPIO × BIOMA × ANO
# ============================================================

COLUNAS_COMPONENTES_CO2E = [
    "emissao_co2e_gwp_ar6_t_direta",
    "emissao_co2e_gwp_ar6_t_indireta"
]


COLUNAS_COMPONENTES_N2O = [
    "emissao_n2o_t_direta",
    "emissao_n2o_t_indireta"
]


seeg_bioma_ano[
    "componentes_co2e_disponiveis"
] = (
    seeg_bioma_ano[
        COLUNAS_COMPONENTES_CO2E
    ]
    .notna()
    .sum(axis=1)
)


seeg_bioma_ano[
    "componentes_n2o_disponiveis"
] = (
    seeg_bioma_ano[
        COLUNAS_COMPONENTES_N2O
    ]
    .notna()
    .sum(axis=1)
)


seeg_bioma_ano[
    "status_dado_seeg_bioma"
] = np.select(
    [
        seeg_bioma_ano[
            "componentes_co2e_disponiveis"
        ].eq(2),

        seeg_bioma_ano[
            "componentes_co2e_disponiveis"
        ].eq(1)
    ],
    [
        "completo",
        "parcial_nao_captado"
    ],
    default="nao_captado"
)


# Soma disponível, mesmo quando parcial
seeg_bioma_ano[
    "emissao_co2e_gwp_ar6_soma_disponivel_t"
] = (
    seeg_bioma_ano[
        COLUNAS_COMPONENTES_CO2E
    ]
    .sum(
        axis=1,
        min_count=1
    )
)


seeg_bioma_ano[
    "emissao_n2o_soma_disponivel_t"
] = (
    seeg_bioma_ano[
        COLUNAS_COMPONENTES_N2O
    ]
    .sum(
        axis=1,
        min_count=1
    )
)


# Total oficial somente quando os dois componentes existem
seeg_bioma_ano[
    "emissao_total_co2e_gwp_ar6_t"
] = (
    seeg_bioma_ano[
        "emissao_co2e_gwp_ar6_soma_disponivel_t"
    ]
    .where(
        seeg_bioma_ano[
            "status_dado_seeg_bioma"
        ].eq(
            "completo"
        )
    )
)


seeg_bioma_ano[
    "emissao_total_n2o_t"
] = (
    seeg_bioma_ano[
        "emissao_n2o_soma_disponivel_t"
    ]
    .where(
        seeg_bioma_ano[
            "status_dado_seeg_bioma"
        ].eq(
            "completo"
        )
    )
)


print(
    "Status:"
)

display(
    seeg_bioma_ano[
        "status_dado_seeg_bioma"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nNulos no total completo CO2e:"
)

print(
    seeg_bioma_ano[
        "emissao_total_co2e_gwp_ar6_t"
    ]
    .isna()
    .sum()
)

Status:


status_dado_seeg_bioma
completo       11511
nao_captado       15
Name: count, dtype: int64


Nulos no total completo CO2e:
15


In [25]:
# ============================================================
# AUDITORIA DOS MUNICÍPIOS COM DADOS NÃO CAPTADOS
# ============================================================

MUNICIPIOS_AUDITORIA = [
    "Cerro Branco",
    "Portão",
    "Venâncio Aires"
]


auditoria_status_seeg = (
    seeg_bioma_ano[
        seeg_bioma_ano[
            "municipio"
        ]
        .isin(
            MUNICIPIOS_AUDITORIA
        )
    ]
    [
        [
            "municipio",
            "estado",
            "bioma",
            "ano",
            "emissao_co2e_gwp_ar6_t_direta",
            "emissao_co2e_gwp_ar6_t_indireta",
            "emissao_co2e_gwp_ar6_soma_disponivel_t",
            "emissao_total_co2e_gwp_ar6_t",
            "status_dado_seeg_bioma"
        ]
    ]
    .sort_values(
        [
            "municipio",
            "bioma",
            "ano"
        ]
    )
)


display(
    auditoria_status_seeg
)

,municipio,estado,bioma,ano,emissao_co2e_gwp_ar6_t_direta,emissao_co2e_gwp_ar6_t_indireta,emissao_co2e_gwp_ar6_soma_disponivel_t,emissao_total_co2e_gwp_ar6_t,status_dado_seeg_bioma
2268,Cerro Branco,RS,Mata Atlântica,2019,282.95,63.66,346.61,346.61,completo
2269,Cerro Branco,RS,Mata Atlântica,2020,167.44,37.68,205.12,205.12,completo
2270,Cerro Branco,RS,Mata Atlântica,2021,332.56,74.83,407.39,407.39,completo
2271,Cerro Branco,RS,Mata Atlântica,2022,273.49,61.53,335.02,335.02,completo
2272,Cerro Branco,RS,Mata Atlântica,2023,182.32,41.02,223.34,223.34,completo
2273,Cerro Branco,RS,Mata Atlântica,2024,312.63,70.34,382.97,382.97,completo
2274,Cerro Branco,RS,Pampa,2019,NaN,NaN,NaN,NaN,nao_captado
2275,Cerro Branco,RS,Pampa,2020,NaN,NaN,NaN,NaN,nao_captado
2276,Cerro Branco,RS,Pampa,2021,NaN,NaN,NaN,NaN,nao_captado
2277,Cerro Branco,RS,Pampa,2022,0.01,0.00,0.01,0.01,completo


In [26]:
# ============================================================
# CONTEXTO TERRITORIAL — BIOMAS POR MUNICÍPIO
# ============================================================

biomas_por_municipio_seeg = (
    seeg_bioma_ano
    .groupby(
        [
            "municipio",
            "estado"
        ],
        as_index=False
    )
    .agg(
        quantidade_biomas_seeg=(
            "bioma",
            "nunique"
        ),

        biomas_presentes_seeg=(
            "bioma",
            lambda serie:
                " | ".join(
                    sorted(
                        serie
                        .dropna()
                        .unique()
                    )
                )
        )
    )
)


print(
    "Municípios:"
)

print(
    len(
        biomas_por_municipio_seeg
    )
)


print(
    "\nQuantidade de biomas por município:"
)

display(
    biomas_por_municipio_seeg[
        "quantidade_biomas_seeg"
    ]
    .value_counts()
    .sort_index()
)


display(
    biomas_por_municipio_seeg.head()
)

Municípios:
1658

Quantidade de biomas por município:


quantidade_biomas_seeg
1    1396
2     261
3       1
Name: count, dtype: int64

,municipio,estado,quantidade_biomas_seeg,biomas_presentes_seeg
0,Abadia de Goiás,GO,1,Cerrado
1,Abadiânia,GO,1,Cerrado
2,Abatiá,PR,1,Mata Atlântica
3,Abdon Batista,SC,1,Mata Atlântica
4,Abelardo Luz,SC,1,Mata Atlântica


In [27]:
# ============================================================
# CONSOLIDAÇÃO — MUNICÍPIO × ANO
# ============================================================

CHAVE_SEEG_MUNICIPIO_ANO = [
    "municipio",
    "estado",
    "ano"
]


COLUNAS_SOMA_MUNICIPAL = [
    "emissao_co2e_gwp_ar6_t_direta",
    "emissao_co2e_gwp_ar6_t_indireta",
    "emissao_n2o_t_direta",
    "emissao_n2o_t_indireta",
    "emissao_co2e_gwp_ar6_soma_disponivel_t",
    "emissao_n2o_soma_disponivel_t"
]


# ------------------------------------------------------------
# Somatórios dos valores disponíveis
# ------------------------------------------------------------

seeg_somas_municipio_ano = (
    seeg_bioma_ano
    .groupby(
        CHAVE_SEEG_MUNICIPIO_ANO
    )[
        COLUNAS_SOMA_MUNICIPAL
    ]
    .sum(
        min_count=1
    )
    .reset_index()
)


# ------------------------------------------------------------
# Auditoria de completude dos biomas
# ------------------------------------------------------------

seeg_status_municipio_ano = (
    seeg_bioma_ano
    .groupby(
        CHAVE_SEEG_MUNICIPIO_ANO,
        as_index=False
    )
    .agg(
        quantidade_biomas_ano=(
            "bioma",
            "nunique"
        ),

        biomas_completos=(
            "status_dado_seeg_bioma",
            lambda serie:
                (
                    serie == "completo"
                ).sum()
        ),

        biomas_parciais=(
            "status_dado_seeg_bioma",
            lambda serie:
                (
                    serie == "parcial_nao_captado"
                ).sum()
        ),

        biomas_nao_captados=(
            "status_dado_seeg_bioma",
            lambda serie:
                (
                    serie == "nao_captado"
                ).sum()
        )
    )
)


# ------------------------------------------------------------
# Juntar somas + auditoria + contexto de biomas
# ------------------------------------------------------------

seeg_municipio_ano = (
    seeg_somas_municipio_ano
    .merge(
        seeg_status_municipio_ano,
        on=CHAVE_SEEG_MUNICIPIO_ANO,
        how="left",
        validate="one_to_one"
    )
    .merge(
        biomas_por_municipio_seeg,
        on=[
            "municipio",
            "estado"
        ],
        how="left",
        validate="many_to_one"
    )
)


print(
    "Dimensão:"
)

print(
    seeg_municipio_ano.shape
)


print(
    "\nMunicípios:"
)

print(
    seeg_municipio_ano[
        [
            "municipio",
            "estado"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


print(
    "\nDuplicatas município + estado + ano:"
)

print(
    seeg_municipio_ano
    .duplicated(
        subset=CHAVE_SEEG_MUNICIPIO_ANO
    )
    .sum()
)

Dimensão:
(9948, 15)

Municípios:
1658

Duplicatas município + estado + ano:
0


In [28]:
# ============================================================
# STATUS E TOTAL — MUNICÍPIO × ANO
# ============================================================

seeg_municipio_ano[
    "status_dado_seeg"
] = np.select(
    [
        (
            seeg_municipio_ano[
                "biomas_completos"
            ]
            ==
            seeg_municipio_ano[
                "quantidade_biomas_seeg"
            ]
        ),

        (
            seeg_municipio_ano[
                "biomas_completos"
            ] > 0
        )
    ],
    [
        "completo",
        "parcial_nao_captado"
    ],
    default="nao_captado"
)


# ------------------------------------------------------------
# Renomear somas disponíveis
# ------------------------------------------------------------

seeg_municipio_ano = (
    seeg_municipio_ano
    .rename(
        columns={
            "emissao_co2e_gwp_ar6_t_direta":
                "emissao_direta_co2e_gwp_ar6_soma_disponivel_t",

            "emissao_co2e_gwp_ar6_t_indireta":
                "emissao_indireta_co2e_gwp_ar6_soma_disponivel_t",

            "emissao_n2o_t_direta":
                "emissao_direta_n2o_soma_disponivel_t",

            "emissao_n2o_t_indireta":
                "emissao_indireta_n2o_soma_disponivel_t",

            "emissao_co2e_gwp_ar6_soma_disponivel_t":
                "emissao_total_co2e_gwp_ar6_soma_disponivel_t",

            "emissao_n2o_soma_disponivel_t":
                "emissao_total_n2o_soma_disponivel_t"
        }
    )
)


# ------------------------------------------------------------
# Total municipal confiável:
# somente quando TODOS os biomas estão completos
# ------------------------------------------------------------

seeg_municipio_ano[
    "emissao_total_co2e_gwp_ar6_t"
] = (
    seeg_municipio_ano[
        "emissao_total_co2e_gwp_ar6_soma_disponivel_t"
    ]
    .where(
        seeg_municipio_ano[
            "status_dado_seeg"
        ].eq(
            "completo"
        )
    )
)


seeg_municipio_ano[
    "emissao_total_n2o_t"
] = (
    seeg_municipio_ano[
        "emissao_total_n2o_soma_disponivel_t"
    ]
    .where(
        seeg_municipio_ano[
            "status_dado_seeg"
        ].eq(
            "completo"
        )
    )
)


print(
    "Status municipal:"
)

display(
    seeg_municipio_ano[
        "status_dado_seeg"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nNulos no total municipal confiável de CO2e:"
)

print(
    seeg_municipio_ano[
        "emissao_total_co2e_gwp_ar6_t"
    ]
    .isna()
    .sum()
)

Status municipal:


status_dado_seeg
completo               9933
parcial_nao_captado      15
Name: count, dtype: int64


Nulos no total municipal confiável de CO2e:
15


In [29]:
# ============================================================
# REFERÊNCIA TERRITORIAL — CÓDIGO IBGE
# ============================================================

ARQUIVO_REFERENCIA_MUNICIPAL = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "inmet"
    / "municipio_ano"
    / "inmet_municipio_ano_2019_2024.csv"
)


referencia_municipios = pd.read_csv(
    ARQUIVO_REFERENCIA_MUNICIPAL,
    dtype={
        "codigo_ibge": "string"
    }
)


referencia_municipios[
    "codigo_ibge"
] = (
    referencia_municipios[
        "codigo_ibge"
    ]
    .str.strip()
    .str.zfill(7)
)


referencia_municipios = (
    referencia_municipios[
        [
            "codigo_ibge",
            "municipio",
            "uf"
        ]
    ]
    .drop_duplicates()
    .copy()
)


print(
    "Referência municipal:"
)

print(
    referencia_municipios.shape
)


print(
    "\nCódigos IBGE únicos:"
)

print(
    referencia_municipios[
        "codigo_ibge"
    ].nunique()
)


print(
    "\nDuplicatas municipio + uf:"
)

print(
    referencia_municipios
    .duplicated(
        subset=[
            "municipio",
            "uf"
        ]
    )
    .sum()
)


display(
    referencia_municipios.head()
)

Referência municipal:
(1661, 3)

Códigos IBGE únicos:
1661

Duplicatas municipio + uf:
0


,codigo_ibge,municipio,uf
0,5219902,São Francisco de Goiás,GO
1,4316204,Rondinha,RS
2,4317558,Santo Antônio do Palma,RS
3,4209508,Laurentino,SC
4,4202107,Barra Velha,SC


In [30]:
# ============================================================
# PADRONIZAÇÃO AUXILIAR DOS NOMES MUNICIPAIS
# ============================================================

import unicodedata


def normalizar_texto(valor):

    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip().lower()

    valor = unicodedata.normalize(
        "NFKD",
        valor
    )

    valor = "".join(
        caractere
        for caractere in valor
        if not unicodedata.combining(
            caractere
        )
    )

    valor = " ".join(
        valor.split()
    )

    return valor


seeg_municipio_ano[
    "municipio_chave"
] = (
    seeg_municipio_ano[
        "municipio"
    ]
    .apply(
        normalizar_texto
    )
)


seeg_municipio_ano[
    "uf_chave"
] = (
    seeg_municipio_ano[
        "estado"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
)


referencia_municipios[
    "municipio_chave"
] = (
    referencia_municipios[
        "municipio"
    ]
    .apply(
        normalizar_texto
    )
)


referencia_municipios[
    "uf_chave"
] = (
    referencia_municipios[
        "uf"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [31]:
# ============================================================
# AUDITORIA DA CHAVE MUNICIPAL NORMALIZADA
# ============================================================

print(
    "Duplicatas na referência após normalização:"
)

print(
    referencia_municipios
    .duplicated(
        subset=[
            "municipio_chave",
            "uf_chave"
        ]
    )
    .sum()
)


duplicatas_referencia = (
    referencia_municipios[
        referencia_municipios
        .duplicated(
            subset=[
                "municipio_chave",
                "uf_chave"
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "uf_chave",
            "municipio_chave"
        ]
    )
)


if len(
    duplicatas_referencia
) > 0:

    display(
        duplicatas_referencia
    )

Duplicatas na referência após normalização:
0


In [32]:
# ============================================================
# ASSOCIAÇÃO DO CÓDIGO IBGE AO SEEG
# ============================================================

referencia_para_merge = (
    referencia_municipios[
        [
            "codigo_ibge",
            "municipio_chave",
            "uf_chave"
        ]
    ]
    .copy()
)


seeg_municipio_ano = (
    seeg_municipio_ano
    .merge(
        referencia_para_merge,
        on=[
            "municipio_chave",
            "uf_chave"
        ],
        how="left",
        validate="many_to_one"
    )
)


print(
    "Dimensão após associação:"
)

print(
    seeg_municipio_ano.shape
)


print(
    "\nRegistros sem código IBGE:"
)

print(
    seeg_municipio_ano[
        "codigo_ibge"
    ]
    .isna()
    .sum()
)


print(
    "\nMunicípios distintos sem código:"
)

print(
    seeg_municipio_ano.loc[
        seeg_municipio_ano[
            "codigo_ibge"
        ]
        .isna(),
        [
            "municipio",
            "estado"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

Dimensão após associação:
(9948, 21)

Registros sem código IBGE:
0

Municípios distintos sem código:
0


In [33]:
# ============================================================
# AUDITORIA — MUNICÍPIOS SEEG SEM CÓDIGO IBGE
# ============================================================

municipios_seeg_sem_codigo = (
    seeg_municipio_ano.loc[
        seeg_municipio_ano[
            "codigo_ibge"
        ]
        .isna(),
        [
            "municipio",
            "estado",
            "municipio_chave",
            "uf_chave"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "estado",
            "municipio"
        ]
    )
)


print(
    "Municípios sem correspondência:"
)

print(
    len(
        municipios_seeg_sem_codigo
    )
)


if len(
    municipios_seeg_sem_codigo
) > 0:

    display(
        municipios_seeg_sem_codigo
    )

Municípios sem correspondência:
0


In [34]:
# ============================================================
# AUDITORIA TERRITORIAL — REFERÊNCIA IBGE × SEEG
# ============================================================

codigos_seeg = (
    seeg_municipio_ano[
        [
            "codigo_ibge"
        ]
    ]
    .drop_duplicates()
)


territorios_sem_seeg = (
    referencia_municipios[
        [
            "codigo_ibge",
            "municipio",
            "uf"
        ]
    ]
    .merge(
        codigos_seeg.assign(
            presente_seeg=True
        ),
        on="codigo_ibge",
        how="left"
    )
)


territorios_sem_seeg = (
    territorios_sem_seeg[
        territorios_sem_seeg[
            "presente_seeg"
        ]
        .isna()
    ]
    [
        [
            "codigo_ibge",
            "municipio",
            "uf"
        ]
    ]
    .sort_values(
        [
            "uf",
            "municipio"
        ]
    )
)


print(
    "Unidades territoriais da referência:"
)

print(
    referencia_municipios[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nMunicípios presentes no SEEG:"
)

print(
    seeg_municipio_ano[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nUnidades da referência sem registro no SEEG:"
)

print(
    len(
        territorios_sem_seeg
    )
)


display(
    territorios_sem_seeg
)

Unidades territoriais da referência:
1661

Municípios presentes no SEEG:
1658

Unidades da referência sem registro no SEEG:
3


,codigo_ibge,municipio,uf
749,5101837,Boa Esperança do Norte,MT
557,4300001,"Área Operacional ""Lagoa Mirim""",RS
556,4300002,"Área Operacional ""Lagoa dos Patos""",RS


In [35]:
# ============================================================
# CONSTRUÇÃO DA CAMADA CURATED FINAL — SEEG
# ============================================================

seeg_curated = (
    seeg_municipio_ano
    .copy()
)


# ------------------------------------------------------------
# Padronização territorial
# ------------------------------------------------------------

seeg_curated = (
    seeg_curated
    .rename(
        columns={
            "estado": "uf"
        }
    )
)


# ------------------------------------------------------------
# Metadados metodológicos
# ------------------------------------------------------------

seeg_curated[
    "fonte"
] = "SEEG"


seeg_curated[
    "cultura"
] = "Soja"


seeg_curated[
    "setor_emissao"
] = "Agropecuária"


seeg_curated[
    "categoria_emissao"
] = "Solos manejados"


seeg_curated[
    "subcategoria_emissao"
] = "Resíduos agrícolas"


seeg_curated[
    "gas_origem"
] = "N2O"


seeg_curated[
    "metrica_co2e"
] = "GWP-AR6"


# ------------------------------------------------------------
# Seleção e ordenação final das colunas
# ------------------------------------------------------------

COLUNAS_SEEG_CURATED = [
    "codigo_ibge",
    "municipio",
    "uf",
    "ano",
    "cultura",

    "quantidade_biomas_seeg",
    "biomas_presentes_seeg",
    "quantidade_biomas_ano",
    "biomas_completos",
    "biomas_parciais",
    "biomas_nao_captados",
    "status_dado_seeg",

    "emissao_direta_co2e_gwp_ar6_soma_disponivel_t",
    "emissao_indireta_co2e_gwp_ar6_soma_disponivel_t",
    "emissao_total_co2e_gwp_ar6_soma_disponivel_t",
    "emissao_total_co2e_gwp_ar6_t",

    "emissao_direta_n2o_soma_disponivel_t",
    "emissao_indireta_n2o_soma_disponivel_t",
    "emissao_total_n2o_soma_disponivel_t",
    "emissao_total_n2o_t",

    "fonte",
    "setor_emissao",
    "categoria_emissao",
    "subcategoria_emissao",
    "gas_origem",
    "metrica_co2e"
]


seeg_curated = (
    seeg_curated[
        COLUNAS_SEEG_CURATED
    ]
    .sort_values(
        [
            "codigo_ibge",
            "ano"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Dimensão da Curated:"
)

print(
    seeg_curated.shape
)


print(
    "\nColunas:"
)

print(
    seeg_curated.columns.tolist()
)


display(
    seeg_curated.head()
)

Dimensão da Curated:
(9948, 26)

Colunas:
['codigo_ibge', 'municipio', 'uf', 'ano', 'cultura', 'quantidade_biomas_seeg', 'biomas_presentes_seeg', 'quantidade_biomas_ano', 'biomas_completos', 'biomas_parciais', 'biomas_nao_captados', 'status_dado_seeg', 'emissao_direta_co2e_gwp_ar6_soma_disponivel_t', 'emissao_indireta_co2e_gwp_ar6_soma_disponivel_t', 'emissao_total_co2e_gwp_ar6_soma_disponivel_t', 'emissao_total_co2e_gwp_ar6_t', 'emissao_direta_n2o_soma_disponivel_t', 'emissao_indireta_n2o_soma_disponivel_t', 'emissao_total_n2o_soma_disponivel_t', 'emissao_total_n2o_t', 'fonte', 'setor_emissao', 'categoria_emissao', 'subcategoria_emissao', 'gas_origem', 'metrica_co2e']


,codigo_ibge,municipio,uf,ano,cultura,quantidade_biomas_seeg,biomas_presentes_seeg,quantidade_biomas_ano,biomas_completos,biomas_parciais,...,emissao_direta_n2o_soma_disponivel_t,emissao_indireta_n2o_soma_disponivel_t,emissao_total_n2o_soma_disponivel_t,emissao_total_n2o_t,fonte,setor_emissao,categoria_emissao,subcategoria_emissao,gas_origem,metrica_co2e
0,4100103,Abatiá,PR,2019,Soja,1,Mata Atlântica,1,1,0,...,10.45,2.35,12.80,12.80,SEEG,Agropecuária,Solos manejados,Resíduos agrícolas,N2O,GWP-AR6
1,4100103,Abatiá,PR,2020,Soja,1,Mata Atlântica,1,1,0,...,10.35,2.33,12.68,12.68,SEEG,Agropecuária,Solos manejados,Resíduos agrícolas,N2O,GWP-AR6
2,4100103,Abatiá,PR,2021,Soja,1,Mata Atlântica,1,1,0,...,7.87,1.77,9.64,9.64,SEEG,Agropecuária,Solos manejados,Resíduos agrícolas,N2O,GWP-AR6
3,4100103,Abatiá,PR,2022,Soja,1,Mata Atlântica,1,1,0,...,10.52,2.37,12.89,12.89,SEEG,Agropecuária,Solos manejados,Resíduos agrícolas,N2O,GWP-AR6
4,4100103,Abatiá,PR,2023,Soja,1,Mata Atlântica,1,1,0,...,10.18,2.29,12.47,12.47,SEEG,Agropecuária,Solos manejados,Resíduos agrícolas,N2O,GWP-AR6


In [36]:
# ============================================================
# AUDITORIA FINAL — SEEG CURATED
# ============================================================

CHAVE_FINAL_SEEG = [
    "codigo_ibge",
    "ano"
]


print("=" * 70)
print("AUDITORIA FINAL — SEEG")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    seeg_curated.shape
)


print(
    "\nMunicípios distintos:"
)

print(
    seeg_curated[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nAnos:"
)

print(
    sorted(
        seeg_curated[
            "ano"
        ]
        .unique()
        .tolist()
    )
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    seeg_curated
    .duplicated(
        subset=CHAVE_FINAL_SEEG
    )
    .sum()
)


print(
    "\nChaves nulas:"
)

print(
    seeg_curated[
        CHAVE_FINAL_SEEG
    ]
    .isna()
    .sum()
)


print(
    "\nRegistros por ano:"
)

display(
    seeg_curated[
        "ano"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nStatus dos dados:"
)

display(
    seeg_curated[
        "status_dado_seeg"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nNulos — soma disponível CO2e:"
)

print(
    seeg_curated[
        "emissao_total_co2e_gwp_ar6_soma_disponivel_t"
    ]
    .isna()
    .sum()
)


print(
    "\nNulos — total confiável CO2e:"
)

print(
    seeg_curated[
        "emissao_total_co2e_gwp_ar6_t"
    ]
    .isna()
    .sum()
)


print(
    "\nValores negativos CO2e:"
)

print(
    (
        seeg_curated[
            "emissao_total_co2e_gwp_ar6_soma_disponivel_t"
        ] < 0
    )
    .sum()
)


print(
    "\nDiferenças entre quantidade de biomas esperada e anual:"
)

print(
    (
        seeg_curated[
            "quantidade_biomas_seeg"
        ]
        !=
        seeg_curated[
            "quantidade_biomas_ano"
        ]
    )
    .sum()
)

AUDITORIA FINAL — SEEG

Dimensão:
(9948, 26)

Municípios distintos:
1658

Anos:
[2019, 2020, 2021, 2022, 2023, 2024]

Duplicatas codigo_ibge + ano:
0

Chaves nulas:
codigo_ibge    0
ano            0
dtype: int64

Registros por ano:


ano
2019    1658
2020    1658
2021    1658
2022    1658
2023    1658
2024    1658
Name: count, dtype: int64


Status dos dados:


status_dado_seeg
completo               9933
parcial_nao_captado      15
Name: count, dtype: int64


Nulos — soma disponível CO2e:
0

Nulos — total confiável CO2e:
15

Valores negativos CO2e:
0

Diferenças entre quantidade de biomas esperada e anual:
0


In [37]:
# ============================================================
# EXPORTAÇÃO — SEEG MUNICÍPIO × ANO
# ============================================================

SEEG_CURATED_FINAL_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "seeg"
    / "municipio_ano"
)


SEEG_CURATED_FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ARQUIVO_SEEG_CURATED_FINAL = (
    SEEG_CURATED_FINAL_DIR
    / "seeg_emissoes_soja_municipio_ano_2019_2024.csv"
)


seeg_curated.to_csv(
    ARQUIVO_SEEG_CURATED_FINAL,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Arquivo exportado:"
)

print(
    ARQUIVO_SEEG_CURATED_FINAL
)


print(
    "\nArquivo existe:"
)

print(
    ARQUIVO_SEEG_CURATED_FINAL.exists()
)


print(
    "\nTamanho MB:"
)

print(
    round(
        ARQUIVO_SEEG_CURATED_FINAL.stat().st_size
        / (1024 * 1024),
        2
    )
)

Arquivo exportado:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\seeg\municipio_ano\seeg_emissoes_soja_municipio_ano_2019_2024.csv

Arquivo existe:
True

Tamanho MB:
1.86


In [38]:
# ============================================================
# VALIDAÇÃO DO ARQUIVO EXPORTADO
# ============================================================

seeg_curated_validacao = pd.read_csv(
    ARQUIVO_SEEG_CURATED_FINAL,
    dtype={
        "codigo_ibge": "string"
    }
)


seeg_curated_validacao[
    "codigo_ibge"
] = (
    seeg_curated_validacao[
        "codigo_ibge"
    ]
    .str.strip()
    .str.zfill(7)
)


print("=" * 70)
print("VALIDAÇÃO DO CSV EXPORTADO")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    seeg_curated_validacao.shape
)


print(
    "\nMunicípios:"
)

print(
    seeg_curated_validacao[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    seeg_curated_validacao
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


print(
    "\nAnos:"
)

print(
    sorted(
        seeg_curated_validacao[
            "ano"
        ]
        .unique()
        .tolist()
    )
)


print(
    "\nStatus:"
)

display(
    seeg_curated_validacao[
        "status_dado_seeg"
    ]
    .value_counts(
        dropna=False
    )
)

VALIDAÇÃO DO CSV EXPORTADO

Dimensão:
(9948, 26)

Municípios:
1658

Duplicatas codigo_ibge + ano:
0

Anos:
[2019, 2020, 2021, 2022, 2023, 2024]

Status:


status_dado_seeg
completo               9933
parcial_nao_captado      15
Name: count, dtype: int64

In [39]:
# ============================================================
# RESUMO ANUAL DE CONTROLE — SEEG
# ============================================================

resumo_anual_seeg = (
    seeg_curated_validacao
    .groupby(
        "ano",
        as_index=False
    )
    .agg(
        municipios=(
            "codigo_ibge",
            "nunique"
        ),

        registros_completos=(
            "status_dado_seeg",
            lambda serie:
                (
                    serie == "completo"
                ).sum()
        ),

        registros_parciais=(
            "status_dado_seeg",
            lambda serie:
                (
                    serie == "parcial_nao_captado"
                ).sum()
        ),

        co2e_soma_disponivel_t=(
            "emissao_total_co2e_gwp_ar6_soma_disponivel_t",
            "sum"
        ),

        co2e_municipios_completos_t=(
            "emissao_total_co2e_gwp_ar6_t",
            "sum"
        ),

        n2o_soma_disponivel_t=(
            "emissao_total_n2o_soma_disponivel_t",
            "sum"
        )
    )
)


display(
    resumo_anual_seeg
)

,ano,municipios,registros_completos,registros_parciais,co2e_soma_disponivel_t,co2e_municipios_completos_t,n2o_soma_disponivel_t
0,2019,1658,1655,3,8489567.15,8487892.96,31097.31
1,2020,1658,1655,3,8886326.85,8885496.01,32550.75
2,2021,1658,1655,3,9827267.40,9825413.92,35997.49
3,2022,1658,1656,2,8268166.44,8266563.95,30286.51
4,2023,1658,1656,2,10781007.82,10779624.49,39491.02
5,2024,1658,1656,2,10126395.29,10125263.81,37092.93


In [40]:
# ============================================================
# AUDITORIA MATEMÁTICA FINAL
# ============================================================

diferenca_componentes_co2e = (
    (
        seeg_curated_validacao[
            "emissao_direta_co2e_gwp_ar6_soma_disponivel_t"
        ]
        +
        seeg_curated_validacao[
            "emissao_indireta_co2e_gwp_ar6_soma_disponivel_t"
        ]
    )
    -
    seeg_curated_validacao[
        "emissao_total_co2e_gwp_ar6_soma_disponivel_t"
    ]
).abs()


diferenca_componentes_n2o = (
    (
        seeg_curated_validacao[
            "emissao_direta_n2o_soma_disponivel_t"
        ]
        +
        seeg_curated_validacao[
            "emissao_indireta_n2o_soma_disponivel_t"
        ]
    )
    -
    seeg_curated_validacao[
        "emissao_total_n2o_soma_disponivel_t"
    ]
).abs()


print(
    "Maior diferença CO2e:"
)

print(
    diferenca_componentes_co2e.max()
)


print(
    "\nMaior diferença N2O:"
)

print(
    diferenca_componentes_n2o.max()
)


print(
    "\nRegistros CO2e com diferença > 0.000001:"
)

print(
    (
        diferenca_componentes_co2e
        > 0.000001
    )
    .sum()
)


print(
    "\nRegistros N2O com diferença > 0.000001:"
)

print(
    (
        diferenca_componentes_n2o
        > 0.000001
    )
    .sum()
)

Maior diferença CO2e:
2.9103830456733704e-11

Maior diferença N2O:
1.1368683772161603e-13

Registros CO2e com diferença > 0.000001:
0

Registros N2O com diferença > 0.000001:
0


# Conclusão — Tratamento SEEG

O processamento dos dados do SEEG resultou em uma camada analítica
municipal anual compatível com a chave utilizada no Projeto AgroESG.

## Resultado final

A base Curated possui:

- 1.658 municípios;
- período de 2019 a 2024;
- 9.948 registros município-ano;
- chave analítica `codigo_ibge + ano`;
- nenhuma duplicidade na chave;
- nenhuma chave territorial ausente;
- emissões diretas e indiretas preservadas separadamente;
- emissões físicas de N₂O e sua representação em CO₂e GWP-AR6;
- indicadores de completude para registros com dados não captados.

Foram identificados 15 registros município-ano parcialmente captados,
concentrados em Cerro Branco, Portão e Venâncio Aires, no Rio Grande
do Sul. Valores disponíveis desses registros foram preservados, mas
não foram tratados como totais municipais completos.

Três unidades da referência territorial de 2024 não possuem registros
na camada SEEG utilizada:

- Boa Esperança do Norte/MT;
- Área Operacional "Lagoa Mirim"/RS;
- Área Operacional "Lagoa dos Patos"/RS.

Essas ausências foram mantidas como ausência de observação e não foram
convertidas artificialmente em emissões iguais a zero.

## Recorte metodológico

Os valores desta camada correspondem às emissões de N₂O associadas a
resíduos agrícolas da soja em solos manejados, considerando emissões
diretas e indiretas e utilizando GWP-AR6 para representação em CO₂e.

Portanto, os resultados não representam todas as emissões do ciclo de
vida da produção de soja e não correspondem, isoladamente, a créditos
de carbono disponíveis para comercialização.

## Arquivo produzido

`data/databases_curated/seeg/municipio_ano/seeg_emissoes_soja_municipio_ano_2019_2024.csv`

A camada está pronta para integração posterior com as demais bases
Curated do projeto.